# Train Value Transformer

Notebook for training the new scalar-value ViT model. The model predicts one clipped evaluation in pawns for the side to move.

In [ ]:
from __future__ import annotations

import json
import math
import platform
import time
from pathlib import Path

import torch
from torch import nn
from tqdm.auto import tqdm

from transformer_chess import (
    BoardValueTransformer,
    ValueTransformerConfig,
    load_manifest,
    make_dataloader,
    make_optimizer,
    resolve_training_precision,
)


def detect_device(preferred: str = "auto") -> torch.device:
    preferred = preferred.lower()
    mps_backend = getattr(torch.backends, "mps", None)

    if preferred == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        raise RuntimeError("CUDA was requested but is not available.")

    if preferred == "mps":
        if mps_backend is not None and mps_backend.is_available():
            return torch.device("mps")
        raise RuntimeError("MPS was requested but is not available in this process.")

    if preferred == "cpu":
        return torch.device("cpu")

    if torch.cuda.is_available():
        return torch.device("cuda")
    if mps_backend is not None and mps_backend.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def cuda_bf16_supported() -> bool:
    if not torch.cuda.is_available():
        return False
    checker = getattr(torch.cuda, "is_bf16_supported", None)
    if checker is not None:
        return bool(checker())
    major, _ = torch.cuda.get_device_capability()
    return major >= 8


def amp_dtype_for_precision(precision: str) -> torch.dtype | None:
    if precision == "fp16":
        return torch.float16
    if precision == "bf16":
        return torch.bfloat16
    return None


def autocast_kwargs_for_precision(device: torch.device, precision: str) -> tuple[dict[str, object], str]:
    resolved_precision = resolve_training_precision(device, precision)
    kwargs: dict[str, object] = {
        "device_type": device.type,
        "enabled": False,
    }
    autocast_dtype = amp_dtype_for_precision(resolved_precision)
    if autocast_dtype is not None:
        kwargs["enabled"] = True
        kwargs["dtype"] = autocast_dtype
    return kwargs, resolved_precision


def make_grad_scaler(device: torch.device, precision: str) -> torch.amp.GradScaler | None:
    if device.type == "cuda" and precision == "fp16":
        return torch.amp.GradScaler("cuda")
    return None


def resolve_progress_total(loader, max_batches: int | None) -> int | None:
    try:
        total = len(loader)
    except TypeError:
        total = None
    if total is None:
        return max_batches
    if max_batches is None:
        return total
    return min(total, max_batches)


def make_progress_bar(loader, *, max_batches: int | None, desc: str, colour: str):
    return tqdm(
        loader,
        total=resolve_progress_total(loader, max_batches),
        desc=desc,
        leave=False,
        unit="batch",
        dynamic_ncols=True,
        colour=colour,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
    )


def print_runtime_diagnostics() -> None:
    mps_backend = getattr(torch.backends, "mps", None)
    print("platform:", platform.platform())
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    print("cuda_bf16_supported:", cuda_bf16_supported())
    print("cuda_device_count:", torch.cuda.device_count())
    print("mps_backend_present:", mps_backend is not None)
    print("mps_is_built:", mps_backend.is_built() if mps_backend is not None else None)
    print("mps_is_available:", mps_backend.is_available() if mps_backend is not None else None)


def summarize_manifest(dataset_dir: Path) -> None:
    manifest = load_manifest(dataset_dir)
    print("dataset_dir:", dataset_dir)
    print("source:", manifest.source)
    print("samples:", manifest.num_samples)
    print("train:", manifest.num_train)
    print("val:", manifest.num_val)
    print("clip_pawns:", manifest.clip_pawns)
    print("top_k:", manifest.top_k)
    print("min_pvs:", manifest.min_pvs)
    print("num_shards:", len(manifest.shards))


def run_train_epoch_with_progress(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loader,
    *,
    device: torch.device,
    scaler: torch.amp.GradScaler | None,
    precision: str,
    max_batches: int | None = None,
    desc: str = "train",
) -> tuple[float, int, int]:
    criterion = nn.MSELoss()
    autocast_kwargs, _ = autocast_kwargs_for_precision(device, precision)
    model.train()
    total_loss = 0.0
    total_batches = 0
    total_samples = 0
    non_blocking = device.type == "cuda"
    progress = make_progress_bar(loader, max_batches=max_batches, desc=desc, colour="green")
    for batch_index, (inputs, targets) in enumerate(progress, start=1):
        batch_samples = int(inputs.shape[0])
        inputs = inputs.to(device, non_blocking=non_blocking)
        targets = targets.to(device, non_blocking=non_blocking)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(**autocast_kwargs):
            predictions = model(inputs)
            loss = criterion(predictions.float(), targets.float())
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        loss_value = float(loss.detach().item())
        total_loss += loss_value
        total_batches += 1
        total_samples += batch_samples
        progress.set_postfix(batch=f"{batch_index}/{progress.total or '?'}", loss=f"{loss_value:.4f}", avg=f"{total_loss / total_batches:.4f}")
        if max_batches is not None and batch_index >= max_batches:
            break
    if total_batches == 0:
        raise ValueError("No batches were provided for training.")
    progress.close()
    return total_loss / total_batches, total_batches, total_samples


@torch.no_grad()
def run_eval_with_progress(
    model: nn.Module,
    loader,
    *,
    device: torch.device,
    precision: str,
    max_batches: int | None = None,
    desc: str = "val",
) -> tuple[float, int, int]:
    criterion = nn.MSELoss()
    autocast_kwargs, _ = autocast_kwargs_for_precision(device, precision)
    model.eval()
    total_loss = 0.0
    total_batches = 0
    total_samples = 0
    non_blocking = device.type == "cuda"
    progress = make_progress_bar(loader, max_batches=max_batches, desc=desc, colour="blue")
    for batch_index, (inputs, targets) in enumerate(progress, start=1):
        batch_samples = int(inputs.shape[0])
        inputs = inputs.to(device, non_blocking=non_blocking)
        targets = targets.to(device, non_blocking=non_blocking)
        with torch.autocast(**autocast_kwargs):
            predictions = model(inputs)
            loss_value = float(criterion(predictions.float(), targets.float()).item())
        total_loss += loss_value
        total_batches += 1
        total_samples += batch_samples
        progress.set_postfix(batch=f"{batch_index}/{progress.total or '?'}", loss=f"{loss_value:.4f}", avg=f"{total_loss / total_batches:.4f}")
        if max_batches is not None and batch_index >= max_batches:
            break
    if total_batches == 0:
        raise ValueError("No batches were provided for evaluation.")
    progress.close()
    return total_loss / total_batches, total_batches, total_samples


if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print_runtime_diagnostics()


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "value_lichess_1m"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

RUN_MODE = "full"  # "smoke" or "full"
PREFERRED_DEVICE = "auto"  # "auto", "mps", "cpu", "cuda"
PREFERRED_PRECISION = "auto"  # "auto", "fp32", "fp16", "bf16"

PRESETS = {
    "smoke": {
        "epochs": 1,
        "batch_size": 64,
        "num_workers": 0,
        "max_train_batches": 50,
        "max_val_batches": 10,
        "output_name": "value_model_smoke.pt",
    },
    "full": {
        "epochs": 3,
        "batch_size": 256,
        "num_workers": 4,
        "max_train_batches": None,
        "max_val_batches": None,
        "output_name": "value_model_full.pt",
    },
}

MODEL_CONFIG = {
    "d_model": 256,
    "depth": 8,
    "num_heads": 8,
    "mlp_ratio": 4,
    "dropout": 0.1,
}

OPTIMIZER_CONFIG = {
    "lr": 3e-4,
    "weight_decay": 1e-4,
}

SEED = 0
TRAIN_SHUFFLE = True

assert RUN_MODE in PRESETS, f"Unsupported RUN_MODE: {RUN_MODE}"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATASET_DIR =", DATASET_DIR)
print("RUN_MODE =", RUN_MODE)
print("PREFERRED_DEVICE =", PREFERRED_DEVICE)
print("PREFERRED_PRECISION =", PREFERRED_PRECISION)
print("PRESET =", json.dumps(PRESETS[RUN_MODE], indent=2))


In [ ]:
torch.manual_seed(SEED)
device = detect_device(PREFERRED_DEVICE)
resolved_precision = resolve_training_precision(device, PREFERRED_PRECISION)
print("device =", device)
print("resolved_precision =", resolved_precision)

manifest = load_manifest(DATASET_DIR)
summarize_manifest(DATASET_DIR)

preset = PRESETS[RUN_MODE]
train_steps = math.ceil(manifest.num_train / preset["batch_size"])
val_steps = math.ceil(manifest.num_val / preset["batch_size"]) if manifest.num_val else 0
effective_train_steps = min(train_steps, preset["max_train_batches"]) if preset["max_train_batches"] is not None else train_steps
effective_val_steps = min(val_steps, preset["max_val_batches"]) if preset["max_val_batches"] is not None else val_steps

train_loader = make_dataloader(
    DATASET_DIR,
    split="train",
    batch_size=preset["batch_size"],
    num_workers=preset["num_workers"],
    seed=SEED,
    shuffle=TRAIN_SHUFFLE,
)

val_loader = make_dataloader(
    DATASET_DIR,
    split="val",
    batch_size=preset["batch_size"],
    num_workers=preset["num_workers"],
    seed=SEED,
    shuffle=False,
)

config = ValueTransformerConfig(**MODEL_CONFIG)
model = BoardValueTransformer(config).to(device)
optimizer = make_optimizer(model, **OPTIMIZER_CONFIG)
scaler = make_grad_scaler(device, resolved_precision)

print("model_parameters:", sum(p.numel() for p in model.parameters()))
print("batch_size:", preset["batch_size"])
print("num_workers:", preset["num_workers"])
print("train_steps_per_epoch:", effective_train_steps)
print("val_steps_per_epoch:", effective_val_steps)


In [ ]:
history = []
output_path = ARTIFACT_DIR / preset["output_name"]
model.to(device)

for epoch in range(preset["epochs"]):
    print(f"\nEpoch {epoch + 1}/{preset['epochs']}")
    started = time.perf_counter()
    train_loss, train_batches_done, train_samples_done = run_train_epoch_with_progress(
        model,
        optimizer,
        train_loader,
        device=device,
        scaler=scaler,
        precision=resolved_precision,
        max_batches=preset["max_train_batches"],
        desc=f"train {epoch + 1}/{preset['epochs']}",
    )
    val_loss, val_batches_done, val_samples_done = run_eval_with_progress(
        model,
        val_loader,
        device=device,
        precision=resolved_precision,
        max_batches=preset["max_val_batches"],
        desc=f"val {epoch + 1}/{preset['epochs']}",
    )
    elapsed = time.perf_counter() - started
    train_batches_per_second = train_batches_done / elapsed if elapsed > 0 else 0.0
    train_samples_per_second = train_samples_done / elapsed if elapsed > 0 else 0.0
    val_batches_per_second = val_batches_done / elapsed if elapsed > 0 else 0.0
    val_samples_per_second = val_samples_done / elapsed if elapsed > 0 else 0.0

    row = {
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "elapsed_seconds": float(elapsed),
        "train_batches": train_batches_done,
        "train_samples": train_samples_done,
        "train_batches_per_second": float(train_batches_per_second),
        "train_samples_per_second": float(train_samples_per_second),
        "val_batches": val_batches_done,
        "val_samples": val_samples_done,
        "val_batches_per_second": float(val_batches_per_second),
        "val_samples_per_second": float(val_samples_per_second),
        "resolved_precision": resolved_precision,
    }
    history.append(row)
    print(row)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "model_config": MODEL_CONFIG,
            "optimizer_config": OPTIMIZER_CONFIG,
            "run_mode": RUN_MODE,
            "resolved_precision": resolved_precision,
            "device": str(device),
            "history": history,
            "dataset_manifest": load_manifest(DATASET_DIR),
            "clip_pawns": manifest.clip_pawns,
        },
        output_path,
    )

print("saved_checkpoint:", output_path)
